# Lezione 4 — Addestrare il modello

Corso **"Deep Learning con Python e PyTorch da zero"** — DIH DATAlife, basato sulla certificazione EITC/AI/DLPP (European IT Certificate / Artificial Intelligence / Deep Learning with Python and PyTorch).

**Durata stimata della lezione: 120 minuti**

## Obiettivi di apprendimento di questa lezione

- Implementare il ciclo di addestramento completo (forward, backward, step)
- Monitorare la perdita durante l'addestramento e la valutazione
- Riconoscere il sovradattamento (overfitting) e come prevenirlo
- Salvare e ripristinare lo stato del modello

Questo notebook accompagna il capitolo 04 del manuale del corso. Esegui ogni cella in ordine con **Maiusc + Invio**.

## Parte 1 — Preparazione: carica il dataset

Riutilizziamo il setup di lezioni precedenti: MNIST con `DataLoader`.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

print("Versión de PyTorch:", torch.__version__)

## Parte 2 — Definisci il modello

Una rete con uno o due strati nascosti, lo stesso che abbiamo visto nella lezione 3.

In [ ]:
class RedMNIST(nn.Module):
    def __init__(self):
        super().__init__()
        self.capa1 = nn.Linear(28 * 28, 128)
        self.capa2 = nn.Linear(128, 64)
        self.capa3 = nn.Linear(64, 10)

    def forward(self, x):
        x = x.view(x.size(0), -1)   # aplana cada imagen 28x28 en un vector de 784 valores
        x = F.relu(self.capa1(x))
        x = F.relu(self.capa2(x))
        x = self.capa3(x)            # logits en bruto: la función de pérdida se encarga del resto
        return x

red = RedMNIST()
print(red)

## Parte 3 — Funzione di addestramento di un'epoch

Una **epoch** è un passaggio completo attraverso tutto il dataset di addestramento. In ogni batch, il modello fa un passo in avanti (forward), calcola la perdita, si propaga all'indietro (backward) e aggiorna i pesi.

In [ ]:
transformaciones = transforms.Compose([transforms.ToTensor()])

datos_entrenamiento = torchvision.datasets.MNIST(
    root="./datos", train=True, download=True, transform=transformaciones
)
datos_prueba = torchvision.datasets.MNIST(
    root="./datos", train=False, download=True, transform=transformaciones
)

cargador_entrenamiento = torch.utils.data.DataLoader(datos_entrenamiento, batch_size=64, shuffle=True)
cargador_prueba = torch.utils.data.DataLoader(datos_prueba, batch_size=64, shuffle=False)

print("Imágenes de entrenamiento:", len(datos_entrenamiento))
print("Imágenes de prueba:", len(datos_prueba))

## Parte 4 — Funzione di valutazione

Affinché il modello impari davvero i pattern (e non solo memorizzi i dati di addestramento), devi testarlo su dati che non ha mai visto. Se la perdita di addestramento diminuisce ma quella di valutazione aumenta, il modello si sta sovradattando (overfitting).

In [ ]:
funcion_perdida = nn.CrossEntropyLoss()
optimizador = optim.Adam(red.parameters(), lr=0.001)

print(funcion_perdida)
print(optimizador)

## Parte 5 — Il ciclo completo: addestrare per N epoch

Ora che hai le funzioni di addestramento e valutazione, addestrarai il modello per un certo numero di epoch, tracciando la perdita su entrambi gli insiemi (addestramento e validazione) per rilevare il sovradattamento.

In [ ]:
epocas = 5
historial_perdida = []

for epoca in range(epocas):
    perdida_epoca = 0.0
    for imagenes, etiquetas in cargador_entrenamiento:
        optimizador.zero_grad()
        salida = red(imagenes)
        perdida = funcion_perdida(salida, etiquetas)
        perdida.backward()
        optimizador.step()
        perdida_epoca += perdida.item()

    perdida_media = perdida_epoca / len(cargador_entrenamiento)
    historial_perdida.append(perdida_media)
    print(f"Época {epoca + 1}/{epocas} - pérdida media: {perdida_media:.4f}")

## Parte 6 — Analizzare la convergenza e il sovradattamento

Traccia la perdita di addestramento e validazione nel tempo. Se la perdita di validazione smette di diminuire (o inizia ad aumentare) mentre quella di addestramento continua a scendere, il modello ha smesso di generalizzare.

In [ ]:
plt.plot(range(1, epocas + 1), historial_perdida, marker="o")
plt.xlabel("Época")
plt.ylabel("Pérdida media")
plt.title("Evolución de la pérdida durante el entrenamiento")
plt.show()

## Parte 7 — Salvare e ripristinare il modello

Dopo l'addestramento, vuoi salvare il modello per usarlo in seguito. PyTorch ti consente di salvare solo i pesi (`.state_dict()`) o l'intero modello.

In [ ]:
correctos = 0
total = 0

with torch.no_grad():
    for imagenes, etiquetas in cargador_prueba:
        salida = red(imagenes)
        predicciones = torch.argmax(salida, dim=1)
        correctos += (predicciones == etiquetas).sum().item()
        total += etiquetas.size(0)

precision = correctos / total
print(f"Precisión sobre el conjunto de prueba: {precision:.1%}")

## Riepilogo della lezione

- Un'epoch è un passaggio completo attraverso il dataset di addestramento
- Il ciclo di addestramento è: forward → perdita → backward → step dell'ottimizzatore → zero_grad
- Sempre testare il modello su dati non visti (validation set) per rilevare il sovradattamento
- Se la perdita di validazione non diminuisce mentre quella di addestramento scende, il modello si sta sovradattando: interrompi l'addestramento (early stopping)
- Salva i pesi del modello con `.state_dict()` e ripristinali con `.load_state_dict()`

Nel notebook di pratica caricherai il modello addestrato, continuerai l'addestramento per più epoch e salverai i checkpoint intermedi.

In [ ]:
imagenes, etiquetas = next(iter(cargador_prueba))
with torch.no_grad():
    predicciones = torch.argmax(red(imagenes), dim=1)

figura, ejes = plt.subplots(1, 6, figsize=(12, 3))
for i, eje in enumerate(ejes):
    eje.imshow(imagenes[i][0], cmap="gray")
    eje.set_title(f"real: {etiquetas[i].item()} / pred: {predicciones[i].item()}")
    eje.axis("off")
plt.show()

## Riepilogo della lezione

- Una rete appena creata ha pesi casuali; addestrare è il processo che li aggiusta a partire dai dati
- Il ciclo di addestramento ripete quattro passi per ogni lotto: forward, loss, backward e optimizer.step
- `nn.CrossEntropyLoss` misura l'errore nei problemi di classificazione con più classi; `Adam` e `SGD` sono gli optimizer più comuni per correggerlo
- `optimizador.zero_grad()` prima di ogni `backward()` è indispensabile, perché PyTorch accumula i gradienti per default
- Un'epoca è un passaggio completo sull'insieme di addestramento; ne servono diverse perché la perdita scenda in modo consistente
- La precisione (accuracy) si misura sull'insieme di test, dentro un blocco `torch.no_grad()`, per verificare che la rete generalizzi e non si limiti a memorizzare

Nel notebook di pratica completerai tu stesso un ciclo di addestramento e confronterai gli optimizer.